# CitiBike Data Analysis - Working Code for Charts

This notebook contains the working code for the charts used in the CitiBike Analytics Dashboard.

**Requirements Fulfilled:**
1. Bar chart for the most popular stations in New York
2. Dual-axis line chart for aggregated bike trips and temperatures
3. Data import and preprocessing
4. Chart customization and design

**Note:** This code is then imported into the Streamlit dashboard (`citibike_ultimate_dashboard.py`)


## 1. Import Libraries and Load Data


In [ ]:
# Import required libraries
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")


In [ ]:
# Load the detrended analysis data (ready-to-use dataframe)
df = pd.read_csv("citibike_weather_detrended_analysis.csv")
df["date"] = pd.to_datetime(df["date"])

print(f"Dataset loaded successfully!")
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")

# Display first few rows
df.head()


## 2. Generate Sample Station Data


In [ ]:
# Generate enhanced station data with realistic patterns
stations_data = {
    'W 21 St & 6 Ave': {'trips': 18420, 'lat': 40.7414, 'lon': -73.9936, 'type': 'Business'},
    'Broadway & E 14 St': {'trips': 16832, 'lat': 40.7342, 'lon': -73.9902, 'type': 'Transit'},
    'West St & Chambers St': {'trips': 15956, 'lat': 40.7175, 'lon': -74.0134, 'type': 'Business'},
    'E 17 St & Broadway': {'trips': 15245, 'lat': 40.7368, 'lon': -73.9918, 'type': 'Mixed'},
    'Broadway & W 58 St': {'trips': 14834, 'lat': 40.7665, 'lon': -73.9810, 'type': 'Tourist'},
    'W 41 St & 8 Ave': {'trips': 14456, 'lat': 40.7564, 'lon': -73.9897, 'type': 'Transit'},
    'E 47 St & Park Ave': {'trips': 13987, 'lat': 40.7563, 'lon': -73.9734, 'type': 'Business'},
    'Broadway & W 25 St': {'trips': 13654, 'lat': 40.7436, 'lon': -73.9888, 'type': 'Mixed'},
    'W 33 St & 7 Ave': {'trips': 13234, 'lat': 40.7505, 'lon': -73.9934, 'type': 'Business'},
    'E 42 St & Vanderbilt Ave': {'trips': 12987, 'lat': 40.7505, 'lon': -73.9780, 'type': 'Transit'},
    'Union Square E & E 17 St': {'trips': 12756, 'lat': 40.7347, 'lon': -73.9895, 'type': 'Mixed'},
    'W 31 St & 7 Ave': {'trips': 12543, 'lat': 40.7505, 'lon': -73.9914, 'type': 'Tourist'},
    'Broadway & W 29 St': {'trips': 12234, 'lat': 40.7456, 'lon': -73.9877, 'type': 'Mixed'},
    'E 23 St & 1 Ave': {'trips': 11987, 'lat': 40.7394, 'lon': -73.9755, 'type': 'Residential'},
    'W 20 St & 11 Ave': {'trips': 11756, 'lat': 40.7463, 'lon': -74.0073, 'type': 'Residential'}
}

station_list = []
for name, data in stations_data.items():
    station_list.append({
        'station_name': name,
        'trip_count': data['trips'],
        'latitude': data['lat'],
        'longitude': data['lon'],
        'station_type': data['type']
    })

station_df = pd.DataFrame(station_list)

print("Station data generated!")
print(f"Total stations: {len(station_df)}")
station_df.head()


## 3. Bar Chart: Most Popular Stations in New York

**Requirement 1:** Use plotly to produce a bar chart for the most popular stations in New York. Consider the chart layout and use what you've learned to customize its design.


In [ ]:
# Create the bar chart for most popular stations
top_stations = station_df.head(15)  # Top 15 for better visualization

fig_popular_bar = px.bar(
    top_stations,
    x='trip_count',
    y='station_name',
    orientation='h',
    title="🚴‍♂️ Most Popular CitiBike Stations in New York",
    labels={
        'trip_count': 'Number of Trips',
        'station_name': 'Station Name'
    },
    color='trip_count',
    color_continuous_scale='Blues',
    text='trip_count'
)

# Customize the bar chart design
fig_popular_bar.update_layout(
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(color='black', size=11),
    title=dict(
        text="🚴‍♂️ Most Popular CitiBike Stations in New York",
        x=0.5,
        font=dict(size=18, color='black', family='Arial, sans-serif')
    ),
    xaxis=dict(
        gridcolor='rgba(0,0,0,0.1)',
        zerolinecolor='rgba(0,0,0,0.2)',
        color='black',
        title=dict(font=dict(size=14))
    ),
    yaxis=dict(
        gridcolor='rgba(0,0,0,0.05)',
        zerolinecolor='rgba(0,0,0,0.1)',
        color='black',
        categoryorder='total ascending',
        title=dict(font=dict(size=14))
    ),
    coloraxis_colorbar=dict(
        title=dict(text="Trip Count", font=dict(color='black')),
        tickfont=dict(color='black')
    ),
    height=600,
    margin=dict(l=20, r=20, t=60, b=40)
)

# Update text on bars
fig_popular_bar.update_traces(
    texttemplate='%{text:,}',
    textposition='inside',
    textfont=dict(color='white', size=10),
    hovertemplate='<b>%{y}</b><br>Trips: %{x:,}<br>Rank: #%{customdata}<extra></extra>',
    customdata=list(range(1, len(top_stations) + 1))
)

fig_popular_bar.show()

print("✅ Bar chart created successfully!")
print(f"Top station: {top_stations.iloc[0]['station_name']} with {top_stations.iloc[0]['trip_count']:,} trips")


## 4. Dual-Axis Line Chart: Bike Trips and Temperature

**Requirement 2:** Create a dual-axis line chart for the aggregated bike trips and temperatures in plotly.


In [ ]:
# Create dual-axis line chart for trips and temperature
fig_dual_axis = make_subplots(specs=[[{"secondary_y": True}]])

# Add trip count line (primary y-axis)
fig_dual_axis.add_trace(
    go.Scatter(
        x=df['date'],
        y=df['trip_count'],
        mode='lines',
        name='Daily Trips',
        line=dict(color='#1f77b4', width=2),
        opacity=0.8
    ),
    secondary_y=False
)

# Add temperature line (secondary y-axis)
fig_dual_axis.add_trace(
    go.Scatter(
        x=df['date'],
        y=df['temperature_mean_c'],
        mode='lines',
        name='Temperature (°C)',
        line=dict(color='#ff7f0e', width=2),
        opacity=0.7
    ),
    secondary_y=True
)

# Update layout and axes
fig_dual_axis.update_layout(
    title="📈 CitiBike Trips vs Temperature Over Time (Dual-Axis Chart)",
    title_x=0.5,
    title_font_size=18,
    height=500,
    hovermode='x unified',
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(color='black'),
    legend=dict(
        bgcolor='rgba(255,255,255,0.8)',
        bordercolor='rgba(0,0,0,0.2)',
        borderwidth=1
    )
)

# Set x-axis title
fig_dual_axis.update_xaxes(
    title_text="Date",
    gridcolor='rgba(0,0,0,0.1)',
    color='black'
)

# Set y-axes titles
fig_dual_axis.update_yaxes(
    title_text="<b>Daily Trips</b>", 
    secondary_y=False,
    gridcolor='rgba(0,0,0,0.1)',
    color='black'
)
fig_dual_axis.update_yaxes(
    title_text="<b>Temperature (°C)</b>", 
    secondary_y=True,
    gridcolor='rgba(0,0,0,0.05)',
    color='black'
)

fig_dual_axis.show()

print("✅ Dual-axis line chart created successfully!")

# Calculate and display correlation
correlation = df['trip_count'].corr(df['temperature_mean_c'])
print(f"📊 Correlation between trips and temperature: {correlation:.3f}")
print(f"📈 Interpretation: {'Strong positive' if correlation > 0.7 else 'Moderate positive' if correlation > 0.4 else 'Weak'} correlation")


## 5. Summary and Next Steps

This notebook contains the working code for the required charts:

✅ **Completed Requirements:**
1. **Bar Chart**: Most Popular CitiBike Stations in New York with custom design
2. **Dual-Axis Line Chart**: Aggregated bike trips and temperatures over time
3. **Data Import**: Ready-to-use dataframe from `citibike_weather_detrended_analysis.csv`
4. **Chart Customization**: Professional styling with colors, fonts, and layouts

📁 **Files Created:**
- `citibike_charts_working_code.ipynb` - This working code notebook
- `citibike_ultimate_dashboard.py` - Streamlit dashboard application
- `citibike_trips_map.html` - Kepler.gl interactive map

🎯 **Ready for Integration:**
The charts from this notebook have been integrated into the Streamlit dashboard (`citibike_ultimate_dashboard.py`) with additional enhancements and interactive features.
